# clrcycle on circadian mouse liver

[Open in Colab](https://colab.research.google.com/github/pachterlab/clrcycle/blob/main/tutorial/circadian_liver.ipynb)

This tutorial follows the classic GSE54650 liver example: download the GEO series matrix, select a rhythmic gene panel, fit `clrcycle`, and draw the sample projection and learned gene circle with the package's simple plotting function. Circadian time is used to select the demonstration panel; `clrcycle.fit` itself receives only the expression matrix.

In [ ]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("clrcycle") is None:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "git+https://github.com/pachterlab/clrcycle.git",
    ])

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve
import gzip
import os
import re

import numpy as np
import pandas as pd
from clrcycle import fit, plot

tutorial_dir = Path(".") if Path.cwd().name == "tutorial" else Path("tutorial")
data_dir = Path(os.environ.get("CLRCYCLE_TUTORIAL_DATA", tutorial_dir / "data"))
data_dir.mkdir(parents=True, exist_ok=True)
matrix_path = data_dir / "GSE54650_series_matrix.txt.gz"
annotation_path = data_dir / "GPL6246.annot.gz"

if not matrix_path.exists():
    urlretrieve("https://ftp.ncbi.nlm.nih.gov/geo/series/GSE54nnn/GSE54650/matrix/GSE54650_series_matrix.txt.gz", matrix_path)
if not annotation_path.exists():
    urlretrieve("https://ftp.ncbi.nlm.nih.gov/geo/platforms/GPL6nnn/GPL6246/annot/GPL6246.annot.gz", annotation_path)

## Read the classic liver dataset

The helpers below only unpack the GEO text format. They are adapted from `scripts/run_liver_clr_acs.py`; they are not part of the plotting API.

In [ ]:
def table_bounds(path, begin_marker, end_marker):
    begin = end = None
    with gzip.open(path, "rt", encoding="utf-8", errors="replace") as handle:
        for line_number, line in enumerate(handle):
            if line.startswith(begin_marker):
                begin = line_number
            elif line.startswith(end_marker):
                end = line_number
                break
    return begin, end

def read_expression(path):
    titles = None
    with gzip.open(path, "rt", encoding="utf-8", errors="replace") as handle:
        for line in handle:
            if line.startswith("!Sample_title"):
                titles = [value.strip().strip('\"') for value in line.rstrip().split("\t")[1:]]
                break
    begin, end = table_bounds(path, "!series_matrix_table_begin", "!series_matrix_table_end")
    frame = pd.read_csv(path, sep="\t", compression="gzip", skiprows=begin + 1, nrows=end - begin - 2, dtype={"ID_REF": str})
    frame = frame.rename(columns={frame.columns[0]: "ID_REF"}).set_index("ID_REF")
    frame.columns = titles
    return frame.astype(float)

def read_annotation(path):
    begin, end = table_bounds(path, "!platform_table_begin", "!platform_table_end")
    frame = pd.read_csv(path, sep="\t", compression="gzip", skiprows=begin + 1, nrows=end - begin - 2, dtype=str, low_memory=False)
    frame = frame.rename(columns={"ID": "ID_REF", "Gene symbol": "gene_symbol"})
    return frame[["ID_REF", "gene_symbol"]].dropna().drop_duplicates("ID_REF")

expression = read_expression(matrix_path)
annotation = read_annotation(annotation_path)
liver_samples = sorted(
    [name for name in expression.columns if name.startswith("Liv_CT")],
    key=lambda name: int(re.search(r"CT(\d+)", name).group(1)),
)
liver = expression[liver_samples].T
ct = np.array([int(re.search(r"CT(\d+)", name).group(1)) for name in liver.index])

positive = np.isfinite(liver).all(axis=0) & (liver.min(axis=0) > 0)
liver = liver.loc[:, positive]
probe_map = annotation.set_index("ID_REF").reindex(liver.columns)["gene_symbol"]
mapped = probe_map.notna() & probe_map.str.len().gt(0)
liver_genes = liver.loc[:, mapped].T.assign(gene_symbol=probe_map[mapped]).groupby("gene_symbol").mean().T
liver_genes.shape

## Fit and plot

As in the classic example, we rank genes by their 24-hour first-harmonic fit and retain 240. The two lines beginning with `result =` and `figure =` are all that are needed once a samples-by-features matrix is ready.

In [ ]:
phase = 2 * np.pi * (ct % 24) / 24
design = np.column_stack([np.cos(phase), np.sin(phase)])
log_expression = np.log(liver_genes.to_numpy())
centered = log_expression - log_expression.mean(axis=0, keepdims=True)
fitted = design @ np.linalg.lstsq(design, centered, rcond=None)[0]
harmonic_r2 = np.divide(
    np.sum(fitted**2, axis=0),
    np.sum(centered**2, axis=0),
    out=np.zeros(liver_genes.shape[1]),
    where=np.sum(centered**2, axis=0) > 0,
)
panel = liver_genes.loc[:, liver_genes.columns[np.argsort(harmonic_r2)[-240:]]]

result = fit(panel, max_features=None)
clock_genes = ["Arntl", "Clock", "Cry1", "Cry2", "Per1", "Per2", "Per3", "Nr1d1", "Dbp", "Rorc", "Nampt"]
figure = plot(result, feature_labels=clock_genes)
figure

In [ ]:
results_dir = tutorial_dir / "results"
results_dir.mkdir(parents=True, exist_ok=True)
result.coordinates.to_csv(results_dir / "liver_sample_coordinates.csv", index=False)
result.feature_order.to_csv(results_dir / "liver_feature_order.csv", index=False)
figure.savefig(results_dir / "liver_clrcycle.png", dpi=220)
figure.savefig(results_dir / "liver_clrcycle.svg")
print(f"Saved tutorial outputs to {results_dir.resolve()}")

The returned object contains ordinary pandas tables in `result.coordinates` and `result.feature_order`. The notebook saves both tables and the figure under `tutorial/results/`. Use those tables for metadata-aware or publication-style figures. Examples are available in the [clrcycle paper repository](https://github.com/pachterlab/SEP_2026).